In [1]:
from dotenv import load_dotenv
from langchain_community.chat_models import ChatOllama

from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr

In [2]:
DB_NAME = "vector_db"
load_dotenv(override=True)
MODEL = "llama3.2:latest"

In [3]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [12]:
retriever = vectorstore.as_retriever()


llm = ChatOllama(
    model="llama3.2:latest",
    temperature=1,
    base_url="http://localhost:11434"
)

In [13]:
retriever.invoke("Who is Avery?")

[Document(id='a0e46ef0-b870-4465-8f1a-d59f5f0f3f46', metadata={'doc_type': 'employees', 'source': 'knowledge-base/employees/Avery Lancaster.md'}, page_content="## Other HR Notes\n- **Professional Development**: Avery has actively participated in leadership training programs and industry conferences, representing Insurellm and fostering partnerships.  \n- **Diversity & Inclusion Initiatives**: Avery has championed a commitment to diversity in hiring practices, seeing visible improvements in team representation since 2021.  \n- **Work-Life Balance**: Feedback revealed concerns regarding work-life balance, which Avery has approached by implementing flexible working conditions and ensuring regular check-ins with the team.\n- **Community Engagement**: Avery led community outreach efforts, focusing on financial literacy programs, particularly aimed at underserved populations, improving Insurellm's corporate social responsibility image.  \n\nAvery Lancaster has demonstrated resilience and ada

In [14]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [15]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [16]:
answer_question("Who is Averi Lancaster?", [])

'Avery Lancaster is the Co-Founder & Chief Executive Officer (CEO) of Insurellm. She co-founded the company in 2015 and has been instrumental in guiding it to its current position as a leading Insurance Tech provider. Under her leadership, Insurellm has been able to innovate and succeed in the mainstream insurance market.'

In [17]:
gr.ChatInterface(answer_question).launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


## ✅ Result

- Successfully implemented Retrieval-Augmented Generation (RAG)
- Model answers are now grounded in the knowledge base
- Hallucinations reduced by enforcing context-only responses

### Example

**Query:** Who is Avery?

**Answer:**  
Avery Lancaster is the Co-Founder and Chief Executive Officer (CEO) of Insurellm...